### Procesamiento de Lenguaje Natural I
# **Desafío 1**

FEDERICO AGUSTIN FERNANDEZ DE FRANCESCO
DNI 29433385
N: a2522


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [1]:
%pip install numpy scikit-learn -q


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score
import numpy as np

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [5]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [6]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [7]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [8]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [9]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [10]:
# tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [11]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [12]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [13]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [14]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [15]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [16]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [17]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  1534, 10055,  4750], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [18]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [19]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [20]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [21]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [22]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [23]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**


**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## 1. Vectorización de documentos y similaridad entre documentos

Elegimos 5 documentos al azar (con una semilla fija para que el experimento sea reproducible) y, para cada uno, calculamos la similaridad coseno contra **todo** el conjunto de entrenamiento usando la matriz `X_train` (TF-IDF) ya calculada arriba. Luego miramos los 5 documentos más parecidos a cada uno (excluyendo el propio documento) y comparamos:

- El texto (para ver si el contenido es realmente similar).
- La clase (`target_names`) del documento elegido vs. la clase de sus vecinos más similares.

Esto nos permite evaluar, de forma cualitativa, si la representación TF-IDF + similaridad coseno agrupa correctamente documentos de la misma temática.

In [24]:
SEED = 42
rng = np.random.default_rng(SEED)
random_idxs = rng.choice(X_train.shape[0], size=5, replace=False)
print('Índices de documentos elegidos al azar:', random_idxs)

Índices de documentos elegidos al azar: [8754 4965 7404 1009 4899]


In [25]:
def mostrar_similares(idx, X, textos, y, target_names, top_k=5, chars=200):
    """Calcula la similaridad coseno del documento `idx` contra toda la matriz X
    y muestra los `top_k` documentos más similares (excluyendo el propio documento)."""
    cossim = cosine_similarity(X[idx], X)[0]
    mostsim = np.argsort(cossim)[::-1][1:top_k + 1]  # se descarta la posición 0 (el propio documento)

    print('=' * 100)
    print(f'DOCUMENTO {idx}  |  clase real: {target_names[y[idx]]}')
    print('-' * 100)
    print(textos[idx][:chars].replace(chr(10), ' '), '...')
    print()
    print(f'Top {top_k} documentos más similares:')
    for rank, i in enumerate(mostsim, start=1):
        print(f'  {rank}) doc {i:5d} | similaridad={cossim[i]:.3f} | clase: {target_names[y[i]]}')
        print(f'      texto: {textos[i][:chars].strip().replace(chr(10), " ")} ...')
    print()
    return mostsim, cossim

In [26]:
resultados_similares = {}
for idx in random_idxs:
    mostsim, cossim = mostrar_similares(
        idx, X_train, newsgroups_train.data, y_train, newsgroups_train.target_names
    )
    resultados_similares[idx] = (mostsim, cossim)

DOCUMENTO 8754  |  clase real: talk.religion.misc
----------------------------------------------------------------------------------------------------
 /(hudson) /If someone inflicts pain on themselves, whether they enjoy it or not, they /are hurting themselves.  They may be permanently damaging their body.  That is true.  It is also none of your bu ...

Top 5 documentos más similares:
  1) doc  6552 | similaridad=0.490 | clase: talk.religion.misc
      texto: If I have a habit that I really want to break, and I am willing to make whatever sacrifice I need to make to break it, then I do so. There have been bad habits of mine that I've decided to put forth ...
  2) doc 10613 | similaridad=0.481 | clase: talk.religion.misc
      texto: /(hudson) /Yes you do.  Who is to say that it is immoral for onesself to experience /pain or to be hurt in some other way.  Maybe unpleasant, but that doesn't /say anything about morality.  It violate ...
  3) doc  3616 | similaridad=0.465 | clase: talk.re

In [27]:
# Resumen cuantitativo: ¿cuántos de los 5 vecinos más similares comparten la misma clase
# que el documento de referencia? Esto nos da una idea numérica de qué tan bien
# la similaridad coseno sobre TF-IDF respeta las etiquetas reales.
for idx in random_idxs:
    mostsim, _ = resultados_similares[idx]
    clase_doc = y_train[idx]
    coincidencias = sum(1 for i in mostsim if y_train[i] == clase_doc)
    print(f'Doc {idx} (clase: {newsgroups_train.target_names[clase_doc]}): '
          f'{coincidencias}/5 vecinos comparten la misma clase.')

Doc 8754 (clase: talk.religion.misc): 4/5 vecinos comparten la misma clase.
Doc 4965 (clase: comp.sys.mac.hardware): 2/5 vecinos comparten la misma clase.
Doc 7404 (clase: comp.os.ms-windows.misc): 0/5 vecinos comparten la misma clase.
Doc 1009 (clase: talk.politics.guns): 4/5 vecinos comparten la misma clase.
Doc 4899 (clase: sci.crypt): 3/5 vecinos comparten la misma clase.


### Interpretación (Parte 1)

Con la semilla `SEED=42` salieron sorteados los documentos `[8754, 4965, 7404, 1009, 4899]`. Resultado de coincidencia de clase con sus 5 vecinos más similares:

| Doc | Clase real | Coincidencias (5 vecinos) |
|---|---|---|
| 8754 | talk.religion.misc | 4/5 |
| 4965 | comp.sys.mac.hardware | 2/5 |
| 7404 | comp.os.ms-windows.misc | 0/5 |
| 1009 | talk.politics.guns | 4/5 |
| 4899 | sci.crypt | 3/5 |

**Observaciones:**

- **Doc 8754 (talk.religion.misc, 4/5)**: el texto discute libre albedrío, dolor autoinfligido y moralidad. Sus vecinos más similares (docs 6552, 10613, 3616, 3902) tratan exactamente los mismos temas (hábitos, dolor, moralidad, la Biblia) y son de la misma clase. El único "outlier" (doc 8726, `talk.politics.mideast`, similaridad 0.460) es en realidad un mensaje sobre el desarrollo de un hilo de discusión ("Hasan realized he goofed...") sin vocabulario temático fuerte: probablemente comparte palabras funcionales/de foro más que contenido real, lo que muestra un límite de TF-IDF con mensajes poco temáticos.
- **Doc 4965 (comp.sys.mac.hardware, 2/5)**: el texto es muy corto ("Plug the printer in the printer port, and the modem in the modem port"). Sus 5 vecinos hablan todos de **hardware de puertos seriales/impresoras** (2 de Mac, 2 de IBM PC, 1 de comp.graphics sobre impresoras), es decir, el contenido es coherente aunque la etiqueta específica (Mac vs PC vs graphics) no siempre coincida. Esto tiene sentido: `comp.sys.mac.hardware`, `comp.sys.ibm.pc.hardware` y `comp.graphics` comparten mucho vocabulario técnico de hardware, y un documento tan corto no aporta suficientes palabras "distintivas" de Mac en particular para diferenciarlo del resto del vocabulario de hardware.
- **Doc 7404 (comp.os.ms-windows.misc, 0/5)**: la pregunta es sobre minimizar el "program manager" de Windows. Sus vecinos son de `comp.windows.x` (3), `sci.med` y `comp.graphics`. Este es un caso claro de **ambigüedad de nombres**: "windows" en el vocabulario del vectorizador no distingue entre Microsoft Windows y el sistema de ventanas X11 (`comp.windows.x`), por lo que el modelo agrupa temas relacionados por la palabra pero de dominios distintos. Además el texto es muy corto y genérico ("Hello. Is it possible to...", "please tell me how"), lo que refuerza que dos de los vecinos (docs 8719 y 2429) coincidan solo por compartir el saludo "Hello," y frases de cortesía, no por contenido técnico real.
- **Doc 1009 (talk.politics.guns, 4/5)**: discute regulación de armas y derechos de posesión. Los vecinos usan vocabulario muy específico del debate ("guns", "Waco", "pro-gun", "firearms") y 4 de 5 son de la misma clase; el único distinto (doc 5084, `alt.atheism`) habla de guerras y motivaciones religiosas/políticas, un tema fronterizo entre política y religión que explica el cruce.
- **Doc 4899 (sci.crypt, 3/5)**: el texto menciona a Chomsky y el control de la información, más un registro de opinión/política que técnico de criptografía. Por eso 2 de sus vecinos son de `talk.politics.mideast` y `alt.atheism` (documentos también argumentativos/políticos), mientras que 3 sí son de `sci.crypt` propiamente.

**Conclusión general:** cuando el documento usa vocabulario técnico/temático claro y específico de su grupo (armas, religión, criptografía), la similaridad coseno sobre TF-IDF recupera vecinos de la misma clase de forma consistente (4/5, 4/5, 3/5). Cuando el documento es corto, genérico, o el nombre del grupo es ambiguo respecto al vocabulario (`windows` → MS-Windows vs X-Windows), la coincidencia de clase cae fuertemente (2/5, 0/5), aunque el contenido de los vecinos siga siendo temáticamente razonable. Esto confirma la limitación esperada de un modelo puramente léxico/BoW: no distingue sentidos de una palabra según el contexto (polisemia), y depende de que el documento tenga longitud/vocabulario suficiente para que el vector TF-IDF sea informativo.

## 2. Modelo de clasificación por prototipos (tipo zero-shot / vecino más cercano)

La idea es clasificar cada documento de **test** sin entrenar un modelo probabilístico: para cada documento de test, calculamos su similaridad coseno con **todos** los documentos de entrenamiento y le asignamos la clase del documento de entrenamiento con similaridad máxima (1-Nearest-Neighbor sobre TF-IDF + similaridad coseno).

Como `X_test` tiene ~7500 documentos y `X_train` tiene ~11000, calcular toda la matriz de similaridad de una sola vez (~85 millones de valores) puede consumir mucha memoria. Por eso lo hacemos **en lotes (batches)**: tomamos de a `batch_size` documentos de test, calculamos su similaridad contra todo `X_train`, y nos quedamos con el índice de máxima similaridad (`argmax`) de cada fila.

In [28]:
def clasificar_por_similaridad(X_test, X_train, y_train, batch_size=500):
    """Clasificador tipo zero-shot / 1-NN: para cada documento de X_test devuelve
    la clase del documento de X_train más similar (similaridad coseno)."""
    n_test = X_test.shape[0]
    y_pred = np.empty(n_test, dtype=y_train.dtype)

    for start in range(0, n_test, batch_size):
        end = min(start + batch_size, n_test)
        sims = cosine_similarity(X_test[start:end], X_train)  # (batch_size, n_train)
        vecino_mas_cercano = np.argmax(sims, axis=1)
        y_pred[start:end] = y_train[vecino_mas_cercano]

    return y_pred

In [29]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target

y_pred_zeroshot = clasificar_por_similaridad(X_test, X_train, y_train, batch_size=500)

f1_zeroshot = f1_score(y_test, y_pred_zeroshot, average='macro')
print(f'F1-score Macro (clasificador por prototipos / 1-NN sobre TF-IDF): {f1_zeroshot:.4f}')

F1-score Macro (clasificador por prototipos / 1-NN sobre TF-IDF): 0.5050


In [30]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_zeroshot, target_names=newsgroups_test.target_names, zero_division=0))

                          precision    recall  f1-score   support

             alt.atheism       0.37      0.51      0.43       319
           comp.graphics       0.54      0.48      0.51       389
 comp.os.ms-windows.misc       0.51      0.46      0.48       394
comp.sys.ibm.pc.hardware       0.52      0.52      0.52       392
   comp.sys.mac.hardware       0.53      0.50      0.52       385
          comp.windows.x       0.70      0.59      0.64       395
            misc.forsale       0.63      0.46      0.53       390
               rec.autos       0.41      0.58      0.48       396
         rec.motorcycles       0.63      0.52      0.57       398
      rec.sport.baseball       0.65      0.54      0.59       397
        rec.sport.hockey       0.75      0.72      0.73       399
               sci.crypt       0.55      0.59      0.57       396
         sci.electronics       0.53      0.33      0.41       393
                 sci.med       0.65      0.49      0.56       396
         

### Interpretación (Parte 2)

**F1-macro del clasificador por prototipos (1-NN sobre TF-IDF): 0.5050**

**¿Es mejor que una clasificación aleatoria?** Sí, y por mucho. Con 20 clases, un clasificador que asignara la etiqueta al azar (de forma uniforme) tendría un F1-macro esperado de aproximadamente `1/20 = 0.05`. Incluso un clasificador "aleatorio ponderado" por la frecuencia de cada clase en train seguiría rondando ese mismo orden de magnitud, muy lejos de 0.505. Es decir, el modelo está **10 veces mejor que el azar**, lo que confirma que la representación TF-IDF + similaridad coseno captura información temática real, aun sin entrenar ningún modelo probabilístico.

Mirando el `classification_report`:

- El desempeño es parejo entre clases, con f1 por clase entre ~0.28 (`talk.religion.misc`) y ~0.73 (`rec.sport.hockey`).
- Las clases con vocabulario muy distintivo y acotado (`rec.sport.hockey` f1=0.73, `comp.windows.x` f1=0.64, `rec.motorcycles` f1=0.57) obtienen el mejor desempeño: cuanto más específico el vocabulario de un tema, más "aislado" queda su vector TF-IDF y más fácil es encontrarle vecinos correctos.
- Las clases con peor desempeño son las que en la Parte 1 mostraron solapamiento temático: `talk.religion.misc` (f1=0.28), `talk.politics.misc` (f1=0.31) y `sci.electronics` (f1=0.41). Los grupos `talk.*`, `alt.atheism` y `soc.religion.christian` comparten mucho vocabulario de opinión/debate, por lo que un solo vecino más cercano confunde fácilmente la clase.
- El accuracy global (0.51) es prácticamente igual al F1-macro (0.50), lo que indica que acá el desbalance de clases no es un factor determinante (el dataset está bastante balanceado, ~390-400 documentos de train por clase); las diferencias de desempeño entre clases se deben más a solapamiento temático que a tamaño de clase.

**Debilidades del modelo (por qué queda por debajo de lo que veremos con Naïve Bayes en la Parte 3):**

- **Depende de un único documento de entrenamiento.** La clase asignada es la del *único* vecino más parecido, no un resumen de toda la clase. Si ese documento particular es atípico, corto, o comparte vocabulario "por casualidad" con otro tema (como vimos en la Parte 1 con el doc 7404, clasificado como `comp.windows.x` en vez de `comp.os.ms-windows.misc` por la ambigüedad de la palabra "windows"), la predicción falla aunque el resto de los documentos de la clase correcta sí fueran parecidos.
- **No aprende ninguna distribución de probabilidad por clase.** A diferencia de Naïve Bayes, que agrega estadísticamente la evidencia de *todos* los documentos de entrenamiento de una clase para estimar qué palabras son características de cada tema, el 1-NN no "aprende" nada: solo mide distancia entre vectores. Por eso no puede distinguir, por ejemplo, que una palabra ambigua es más probable en una clase que en otra si solo mira al vecino más cercano.
- **Es sensible a documentos cortos o genéricos.** Cuando el texto de test tiene poco vocabulario temático (saludos, frases de cortesía, mensajes muy breves), su vector TF-IDF queda dominado por palabras poco informativas, y el vecino más cercano puede terminar siendo un documento de otra clase que comparte esas mismas palabras genéricas.
- **Computacionalmente es el método más costoso**: hay que comparar cada documento de test contra los ~11000 documentos de entrenamiento (no hay "modelo" entrenado que resuma la información), mientras que Naïve Bayes, una vez entrenado, predice con una simple multiplicación de probabilidades.

En síntesis: el 1-NN demuestra que la representación TF-IDF por sí sola ya es útil (muy por encima del azar), pero al no agregar evidencia de toda la clase queda por debajo del desempeño que se puede lograr con un modelo que sí aprende una distribución por clase, como se ve en la Parte 3.

## 3. Optimización de modelos Naïve Bayes

Vamos a probar distintas combinaciones de:

- **Vectorizador**: `CountVectorizer` vs `TfidfVectorizer`, variando `max_df`, `min_df`, `sublinear_tf`, `stop_words` (en inglés, ya que el corpus está en inglés).
- **Modelo**: `MultinomialNB` vs `ComplementNB` (este último está pensado específicamente para datasets con clases desbalanceadas, lo cual es relevante para el promediado macro del F1-score).
- **Suavizado (alpha)** de cada modelo.

**Importante**: no modificamos `ngram_range` (se deja en su valor por defecto, `(1, 1)`), tal como pide la consigna.

Entrenamos cada combinación con `newsgroups_train` y evaluamos con `newsgroups_test`, y nos quedamos con la configuración de mayor F1-macro.

In [31]:
from itertools import product

resultados = []

vectorizadores = {
    'tfidf_base': TfidfVectorizer(),
    'tfidf_stopwords': TfidfVectorizer(stop_words='english'),
    'tfidf_stopwords_mindf': TfidfVectorizer(stop_words='english', min_df=2, max_df=0.9),
    'tfidf_sublinear': TfidfVectorizer(stop_words='english', min_df=2, max_df=0.9, sublinear_tf=True),
    'count_stopwords': CountVectorizer(stop_words='english', min_df=2, max_df=0.9),
}

modelos = {
    'MultinomialNB': MultinomialNB,
    'ComplementNB': ComplementNB,
}

alphas = [0.01, 0.05, 0.1, 0.5, 1.0]

for vec_name, vectorizador in vectorizadores.items():
    Xtr = vectorizador.fit_transform(newsgroups_train.data)
    Xte = vectorizador.transform(newsgroups_test.data)

    for model_name, ModelClass in modelos.items():
        for alpha in alphas:
            clf_i = ModelClass(alpha=alpha)
            clf_i.fit(Xtr, y_train)
            y_pred_i = clf_i.predict(Xte)
            f1 = f1_score(y_test, y_pred_i, average='macro')
            resultados.append({
                'vectorizador': vec_name,
                'modelo': model_name,
                'alpha': alpha,
                'f1_macro': f1,
            })

resultados_ordenados = sorted(resultados, key=lambda r: r['f1_macro'], reverse=True)
print('Top 10 combinaciones (vectorizador, modelo, alpha, F1-macro):')
for r in resultados_ordenados[:10]:
    print(f"  {r['vectorizador']:25s} | {r['modelo']:14s} | alpha={r['alpha']:<5} | F1-macro={r['f1_macro']:.4f}")

Top 10 combinaciones (vectorizador, modelo, alpha, F1-macro):
  tfidf_stopwords           | ComplementNB   | alpha=0.5   | F1-macro=0.6978
  tfidf_stopwords_mindf     | ComplementNB   | alpha=0.5   | F1-macro=0.6974
  tfidf_base                | ComplementNB   | alpha=0.5   | F1-macro=0.6961
  tfidf_base                | ComplementNB   | alpha=0.1   | F1-macro=0.6954
  tfidf_sublinear           | ComplementNB   | alpha=0.5   | F1-macro=0.6951
  tfidf_stopwords_mindf     | ComplementNB   | alpha=1.0   | F1-macro=0.6943
  tfidf_stopwords           | ComplementNB   | alpha=1.0   | F1-macro=0.6936
  tfidf_base                | ComplementNB   | alpha=1.0   | F1-macro=0.6930
  tfidf_sublinear           | ComplementNB   | alpha=1.0   | F1-macro=0.6921
  tfidf_stopwords           | ComplementNB   | alpha=0.1   | F1-macro=0.6919


In [32]:
mejor = resultados_ordenados[0]
print('Mejor configuración encontrada:')
print(mejor)

Mejor configuración encontrada:
{'vectorizador': 'tfidf_stopwords', 'modelo': 'ComplementNB', 'alpha': 0.5, 'f1_macro': 0.6978053768076979}


In [33]:
# Reentrenamos y mostramos el reporte de clasificación completo de la mejor configuración
best_vectorizer = vectorizadores[mejor['vectorizador']]
Xtr_best = best_vectorizer.fit_transform(newsgroups_train.data)
Xte_best = best_vectorizer.transform(newsgroups_test.data)

BestModelClass = modelos[mejor['modelo']]
best_clf = BestModelClass(alpha=mejor['alpha'])
best_clf.fit(Xtr_best, y_train)
y_pred_best = best_clf.predict(Xte_best)

print(f"F1-macro final: {f1_score(y_test, y_pred_best, average='macro'):.4f}")
print()
print(classification_report(y_test, y_pred_best, target_names=newsgroups_test.target_names, zero_division=0))

F1-macro final: 0.6978

                          precision    recall  f1-score   support

             alt.atheism       0.31      0.44      0.37       319
           comp.graphics       0.73      0.71      0.72       389
 comp.os.ms-windows.misc       0.72      0.60      0.65       394
comp.sys.ibm.pc.hardware       0.64      0.70      0.67       392
   comp.sys.mac.hardware       0.76      0.72      0.74       385
          comp.windows.x       0.82      0.78      0.80       395
            misc.forsale       0.75      0.72      0.74       390
               rec.autos       0.80      0.75      0.78       396
         rec.motorcycles       0.84      0.78      0.81       398
      rec.sport.baseball       0.92      0.83      0.88       397
        rec.sport.hockey       0.85      0.95      0.90       399
               sci.crypt       0.77      0.80      0.79       396
         sci.electronics       0.71      0.56      0.63       393
                 sci.med       0.81      0.80      

### Interpretación (Parte 3)

**Resultado:** la mejor combinación encontrada fue `TfidfVectorizer(stop_words='english')` + `ComplementNB(alpha=0.5)`, con **F1-macro = 0.6978**, frente al **F1-macro = 0.5854** del Naïve Bayes "de fábrica" (`TfidfVectorizer()` + `MultinomialNB()` sin ajustar) del ejemplo inicial de la notebook. Es una mejora de **+11.2 puntos de F1-macro**.

Mirando el top 10 de combinaciones:

- **Las 10 mejores combinaciones son todas con `ComplementNB`**, ninguna con `MultinomialNB` entra al top 10. Esto confirma lo esperado en la teoría: `ComplementNB` está diseñado para compensar el sesgo hacia las clases con más ejemplos de entrenamiento, y como el F1-**macro** pondera todas las clases por igual (aunque acá el dataset esté bastante balanceado, siguen existiendo diferencias de tamaño y de "dificultad" entre clases, como vimos en la Parte 2), `ComplementNB` saca ventaja justamente en esas clases más difíciles.
- **Quitar stopwords en inglés ayuda, pero poco**: `tfidf_stopwords` (0.6978) es apenas superior a `tfidf_base` (0.6961) con el mismo alpha. Esto tiene sentido porque TF-IDF ya penaliza automáticamente las palabras muy frecuentes (alto DF → bajo IDF), por lo que gran parte del efecto de las stopwords ya estaba "amortiguado" por la propia ponderación TF-IDF, a diferencia de lo que pasaría con un `CountVectorizer` puro.
- **`sublinear_tf=True` no mejoró el resultado** en este caso (0.6951 vs 0.6978 sin él): aplicar logaritmo a las frecuencias de término ayuda cuando hay palabras que se repiten muchísimas veces dentro de un mismo documento, pero en textos relativamente cortos como los de newsgroups ese efecto es menor.
- **El valor de `alpha` óptimo fue 0.5`**, un suavizado moderado: valores más chicos (0.1, 0.01, no mostrados en el top 10 para esta config) ajustan demasiado a las palabras vistas en entrenamiento, y valores más grandes (no se probaron por encima de 1.0) tienden a "aplanar" las diferencias entre clases.
- Comparando el `classification_report` de esta configuración contra el del clasificador por prototipos de la Parte 2: casi todas las clases mejoran su f1 de forma notable (`rec.sport.baseball` pasa de 0.59 a 0.88, `comp.windows.x` de 0.64 a 0.80, `rec.motorcycles` de 0.57 a 0.81). La excepción más marcada es **`talk.religion.misc`**, que baja de 0.28 (1-NN) a **0.22** con Naïve Bayes, y **`alt.atheism`** que se mantiene bajo (0.37 en ambos casos). Esto es consistente con lo visto en la Parte 1: `talk.religion.misc` comparte tanto vocabulario con `alt.atheism` y `soc.religion.christian` que ni el modelo probabilístico logra separarla bien; de hecho el recall de `talk.religion.misc` es muy bajo (0.14), es decir, Naïve Bayes tiende a "regalarle" sus documentos a las otras dos clases religiosas/de opinión.

**Conclusión:** el mayor salto de desempeño no vino de ajustar el vectorizador (los distintos TF-IDF quedan todos muy cerca entre sí, 0.692–0.698), sino de **cambiar el modelo** de `MultinomialNB` a `ComplementNB`. Esto sugiere que, para este dataset y esta tarea, la elección del algoritmo de clasificación pesa más que el preprocesamiento fino del vectorizador (siempre respetando la restricción de no tocar `ngram_range`).

## 4. Similaridad entre palabras (matriz término-documento)

Transponemos la matriz documento-término `X_train` (documentos x términos) para obtener una matriz término-documento (términos x documentos). Cada **fila** de esta matriz transpuesta es ahora un vector que representa a una **palabra** en función de en qué documentos aparece (y con qué peso TF-IDF). Es decir, dos palabras serán "similares" si tienden a aparecer en los mismos documentos.

Elegimos manualmente 5 palabras del vocabulario que sean claramente interpretables (evitando errores de tipeo, fragmentos de emails, números sueltos, etc., que suelen abundar en el vocabulario "crudo" de 20 newsgroups) y buscamos sus 5 vecinas más cercanas según similaridad coseno.

In [34]:
X_train_T = X_train.T  # matriz término-documento (vocab_size x n_docs)
print('Shape de la matriz término-documento:', X_train_T.shape)

Shape de la matriz término-documento: (101631, 11314)


In [35]:
# Elegimos manualmente 5 palabras interpretables y verificamos que existan en el vocabulario
palabras_elegidas = ['car', 'windows', 'god', 'space', 'hockey']

for palabra in palabras_elegidas:
    assert palabra in tfidfvect.vocabulary_, f'"{palabra}" no está en el vocabulario'
print('Todas las palabras elegidas están en el vocabulario. OK')

Todas las palabras elegidas están en el vocabulario. OK


In [36]:
def palabras_similares(palabra, X_T, vocabulario, idx2word, top_k=5):
    idx_palabra = vocabulario[palabra]
    cossim = cosine_similarity(X_T[idx_palabra], X_T)[0]
    mas_similares = np.argsort(cossim)[::-1][1:top_k + 1]  # se descarta la propia palabra

    print(f'Palabra: "{palabra}"')
    for rank, idx in enumerate(mas_similares, start=1):
        print(f'  {rank}) {idx2word[idx]:20s} (similaridad={cossim[idx]:.3f})')
    print()
    return mas_similares

In [37]:
for palabra in palabras_elegidas:
    palabras_similares(palabra, X_train_T, tfidfvect.vocabulary_, idx2word)

Palabra: "car"
  1) cars                 (similaridad=0.180)
  2) criterium            (similaridad=0.177)
  3) civic                (similaridad=0.175)
  4) owner                (similaridad=0.169)
  5) dealer               (similaridad=0.168)

Palabra: "windows"
  1) dos                  (similaridad=0.304)
  2) ms                   (similaridad=0.232)
  3) microsoft            (similaridad=0.222)
  4) nt                   (similaridad=0.214)
  5) for                  (similaridad=0.193)

Palabra: "god"
  1) jesus                (similaridad=0.269)
  2) bible                (similaridad=0.262)
  3) that                 (similaridad=0.256)
  4) existence            (similaridad=0.255)
  5) christ               (similaridad=0.251)

Palabra: "space"
  1) nasa                 (similaridad=0.330)
  2) seds                 (similaridad=0.297)
  3) shuttle              (similaridad=0.293)
  4) enfant               (similaridad=0.280)
  5) seti                 (similaridad=0.246)

Palabra: "

### Interpretación (Parte 4)

Resultados obtenidos (top 5 vecinos por palabra, similaridad coseno sobre la matriz término-documento transpuesta, tamaño del vocabulario: 101.631 términos):

| Palabra | Top 5 más similares |
|---|---|
| `car` | cars (0.180), criterium (0.177), civic (0.175), owner (0.169), dealer (0.168) |
| `windows` | dos (0.304), ms (0.232), microsoft (0.222), nt (0.214), for (0.193) |
| `god` | jesus (0.269), bible (0.262), that (0.256), existence (0.255), christ (0.251) |
| `space` | nasa (0.330), seds (0.297), shuttle (0.293), enfant (0.280), seti (0.246) |
| `hockey` | ncaa (0.274), nhl (0.265), affiliates (0.248), xenophobes (0.243), sportschannel (0.223) |

**Observaciones:**

- Para las 5 palabras se recuperan vecinos **muy coherentes temáticamente**: `car` con `cars`, `civic` (modelo de auto Honda) y `dealer`/`owner`; `windows` con `dos`, `ms`, `microsoft`, `nt` (todos productos/nombres de Microsoft, tal como se esperaba); `god` con `jesus`, `bible`, `christ`; `space` con `nasa`, `shuttle`, `seti` (búsqueda de vida extraterrestre); `hockey` con `ncaa`, `nhl` (las dos ligas/organizaciones de hockey en EE.UU.).
- Las similaridades son **bajas en valor absoluto** (entre 0.17 y 0.33) comparadas con las similaridades entre documentos de la Parte 1 (0.14–0.49). Esto es esperable: cada palabra es un vector de dimensión ~11.000 (una componente por documento), y dos palabras solo tienen valores no nulos donde ambas aparecen; como el vocabulario es enorme y cada palabra aparece en relativamente pocos documentos, el producto punto normalizado da valores moderados aun para palabras claramente relacionadas.
- Aparecen algunos vecinos "ruidosos" fáciles de explicar: `windows` → `for` (una stopword que sobrevivió porque en esta parte usamos `tfidfvect` sin `stop_words='english'`, el vectorizador "base" de la Parte 1); `car` → `criterium` (una carrera ciclística que probablemente se menciona junto a autos en threads de `rec.autos` sobre carreras); `hockey` → `xenophobes` (probablemente aparece en discusiones sobre hockey internacional/olímpico donde se debate nacionalismo). Esto ilustra que la co-ocurrencia a nivel de documento no distingue relaciones semánticas "limpias": basta con que dos palabras aparezcan juntas en varios posts de un mismo hilo de discusión para que queden cerca, aunque no sean sinónimos ni estén conceptualmente emparentadas de forma directa.
- En contraste con lo que se vería con un embedding entrenado por predicción o contexto (Word2Vec, GloVe, BERT — Clase 2), acá la "similaridad" es puramente estadística: dos palabras están cerca si tienden a **compartir documentos**, no porque el modelo haya aprendido su significado o uso sintáctico. Aun así, para palabras suficientemente específicas de un dominio (como las elegidas acá) el resultado es sorprendentemente interpretable, lo que confirma por qué TF-IDF + similaridad coseno sigue siendo una técnica clásica útil como punto de partida en NLP.